# MARL-Gated RALA ViT (v2) — Object Detection on MSCOCO 2017

Fine-tunes Stage 2 (classification pre-trained) model on COCO object detection.

**Key changes from classification v2:**
1. **YOLOS-style**: learnable `[DET]` tokens appended to patch sequence
2. **Hungarian matching** + set prediction loss (L1 + GIoU + CE)
3. **PPO with anti-reward-hacking** — RL for the router with fixed reward:
   - Variance maximization bonus: encourages spatial differentiation
   - Bimodal sparsity penalty: `w*(1-w)` pushes routing toward 0 or 1
   - Symmetric budget: `(mean-target)^2` replaces one-sided `ReLU`
4. **Stage 2 backbone+router weights** transfer via `strict=False` loading

**Dataset:** MSCOCO 2017 via `kagglehub`

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Normal
from torchvision import transforms
from torchvision.datasets import CocoDetection
from torchvision.ops import generalized_box_iou
from einops import rearrange
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment
import kagglehub
import math
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

## 1. Architecture (Backbone from marl_vit_v2)

In [ ]:
class SharedActorCritic(nn.Module):
    """
    Shared Actor-Critic for the swarm of N agents.
    Each token is an independent agent; the AC processes them all in parallel.
    The actor outputs a squashed Gaussian scalar weight w in (0,1).
    The critic outputs a per-token value estimate.
    """
    def __init__(self, d_model: int):
        super().__init__()
        half = max(d_model // 2, 16)
        self.proj = nn.Linear(d_model, half)
        self.actor_mlp = nn.Sequential(nn.Linear(half, half), nn.ELU())
        self.mu_head = nn.Linear(half, 1)
        self.sigma_head = nn.Linear(half, 1)
        self.critic_mlp = nn.Sequential(
            nn.Linear(half, half), nn.ELU(), nn.Linear(half, 1)
        )

    def forward(self, local_features, deterministic=False):
        s = self.proj(local_features)
        h_actor = self.actor_mlp(s)
        mu = self.mu_head(h_actor).squeeze(-1)
        sigma = F.softplus(self.sigma_head(h_actor)).squeeze(-1) + 1e-5
        value = self.critic_mlp(s).squeeze(-1)

        if deterministic:
            z = mu
            log_prob = None
        else:
            dist = Normal(mu, sigma)
            z = dist.rsample()
            log_prob = dist.log_prob(z)

        w_raw = torch.tanh(z)
        w = (w_raw + 1.0) / 2.0

        if not deterministic:
            jacobian = torch.log(0.5 * (1.0 - w_raw.pow(2)) + 1e-5)
            log_prob = log_prob - jacobian

        return w, log_prob, value, mu, sigma

In [ ]:
class ChunkwiseRALAAttention(nn.Module):
    """Chunkwise Linear Attention with MARL gating."""
    def __init__(self, d_model, head=8, chunk_size=16, gamma=0.1, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.head = head
        self.chunk_size = chunk_size
        self.gamma = gamma
        self.d_k = d_model // head
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o_gate = nn.Linear(d_model, d_model)
        self.w_o_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, w_gating, use_dilution=False):
        b, n, d = x.shape
        T = n // self.chunk_size
        C = self.chunk_size

        q = rearrange(self.w_q(x), 'b (T C) (h dk) -> b T h C dk', T=T, C=C, h=self.head)
        k = rearrange(self.w_k(x), 'b (T C) (h dk) -> b T h C dk', T=T, C=C, h=self.head)
        v = rearrange(self.w_v(x), 'b (T C) (h dk) -> b T h C dk', T=T, C=C, h=self.head)

        q = q * (self.d_k ** -0.25)
        k = k * (self.d_k ** -0.25)
        phi_q = F.elu(q) + 1.0
        phi_k = F.elu(k) + 1.0

        w_chunks = rearrange(w_gating, 'b (T C) -> b T C', T=T, C=C)
        w_expanded = w_chunks.unsqueeze(2).unsqueeze(-1)
        k_gated = w_expanded * phi_k

        k_gated_f32 = k_gated.to(torch.float32)
        v_f32 = v.to(torch.float32)
        KV_chunks = torch.matmul(k_gated_f32.transpose(-2, -1), v_f32)
        Z_chunks = k_gated_f32.sum(dim=-2)
        w_bar = w_chunks.mean(dim=-1)

        outputs = []
        S = torch.zeros(b, self.head, self.d_k, self.d_k, device=x.device, dtype=torch.float32)
        Z = torch.zeros(b, self.head, self.d_k, device=x.device, dtype=torch.float32)

        for t in range(T):
            decay_factor = 1.0 - (self.gamma * (1.0 - w_bar[:, t]))
            decay_S = decay_factor.view(b, 1, 1, 1)
            decay_Z = decay_factor.view(b, 1, 1)

            if use_dilution and t > 0:
                dilution_scale = self.gamma * (1.0 - w_bar[:, t])
                gamma_tau = dilution_scale.view(b, 1, 1, 1) * S / max(t, 1)
                S = (S * decay_S) + KV_chunks[:, t] + gamma_tau
            else:
                S = (S * decay_S) + KV_chunks[:, t]

            Z = (Z * decay_Z) + Z_chunks[:, t]
            phi_q_t = phi_q[:, t].to(torch.float32)
            nom = torch.matmul(phi_q_t, S)
            denom = (phi_q_t * Z.unsqueeze(-2)).sum(dim=-1, keepdim=True) + 1e-5
            out_t = nom / denom
            if self.training:
                out_t = self.dropout(out_t)
            out_t = torch.clamp(out_t, min=-65000.0, max=65000.0)
            outputs.append(out_t.to(q.dtype))

        out = torch.stack(outputs, dim=1)
        out = rearrange(out, 'b T h C dk -> b (T C) (h dk)')
        gate = torch.sigmoid(self.w_o_gate(x))
        out = out * gate
        out = self.w_o_proj(out)
        return out, phi_k


class PatchEmbedding(nn.Module):
    def __init__(self, image_size=144, patch_size=12, in_chans=3, embed_dim=256):
        super().__init__()
        self.num_patches = (image_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


class MLP(nn.Module):
    def __init__(self, in_features, hidden_features, drop=0.):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, in_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class EncoderBlock(nn.Module):
    def __init__(self, d_model, head, chunk_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = ChunkwiseRALAAttention(d_model, head=head, chunk_size=chunk_size)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, d_model * 4)

    def forward(self, x, w_gating, use_dilution=False):
        res = x
        x_normed = self.norm1(x)
        out, phi_k = self.attn(x_normed, w_gating, use_dilution)
        x = res + out
        x = x + self.mlp(self.norm2(x))
        return x, phi_k

## 2. Detection Model — YOLOS-style with MARL Routing

- **32 learnable `[DET]` tokens** appended to 144 patch tokens (total = 176 = 11 x 16 chunks)
- **Router only gates patch tokens** — detection tokens always get `w=1.0`
- **cls_head**: 2-layer MLP -> 81 classes (80 COCO + 1 no-object)
- **bbox_head**: 3-layer MLP + sigmoid -> 4 normalized coords (cx, cy, w, h)

In [ ]:
class ViTDetection(nn.Module):
    """
    MARL-Gated RALA ViT for Object Detection.

    - Learnable [DET] tokens appended to the patch sequence
    - Detection heads (class + bbox) on [DET] token outputs
    - [DET] tokens always get w=1.0 (fully retained)
    - Router only gates patch tokens -> learns which regions matter for objects
    """
    def __init__(self, image_size=144, patch_size=12,
                 d_model=256, depth=16, head=8, chunk_size=16,
                 num_det_tokens=32, num_det_classes=91):
        super().__init__()
        self.depth = depth
        self.d_model = d_model
        self.num_det_tokens = num_det_tokens
        self.num_det_classes = num_det_classes

        # --- Backbone (same param names as classification ViT for weight loading) ---
        self.patch_embed = PatchEmbedding(image_size, patch_size, 3, d_model)
        self.num_patches = self.patch_embed.num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, d_model))
        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, head, chunk_size) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(d_model)

        # --- Shared router (same as classification ViT) ---
        self.router = SharedActorCritic(d_model)

        # --- Detection-specific layers ---
        self.det_tokens = nn.Parameter(torch.randn(1, num_det_tokens, d_model) * 0.02)
        self.cls_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, num_det_classes))
        self.bbox_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.ReLU(),
            nn.Linear(d_model, d_model), nn.ReLU(),
            nn.Linear(d_model, 4))

    def load_backbone_weights(self, checkpoint_path, device='cpu'):
        """Load Stage 2 classification weights. Skips old cls head, keeps det layers random."""
        state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
        filtered = {k: v for k, v in state_dict.items() if not k.startswith('head.')}
        missing, unexpected = self.load_state_dict(filtered, strict=False)
        print(f"Loaded backbone+router from {checkpoint_path}")
        print(f"   Missing (new det layers): {len(missing)} | Unexpected (skipped): {len(unexpected)}")
        return missing, unexpected

    def forward(self, x, deterministic=False, phase1=False, use_dilution=True):
        B = x.shape[0]
        N_patch = self.num_patches
        N_det = self.num_det_tokens

        x_patch = self.patch_embed(x) + self.pos_embed
        det = self.det_tokens.expand(B, -1, -1)
        x = torch.cat([x_patch, det], dim=1)

        w_list, log_prob_list, value_list, mu_list, sigma_list = [], [], [], [], []

        for block in self.blocks:
            w, log_prob, value, mu, sigma = self.router(x[:, :N_patch], deterministic)
            w_patch = torch.ones_like(w) if phase1 else w
            w_det = torch.ones(B, N_det, device=x.device, dtype=w_patch.dtype)
            w_gating = torch.cat([w_patch, w_det], dim=1)
            x, _ = block(x, w_gating, use_dilution)

            w_list.append(w)
            log_prob_list.append(log_prob)
            value_list.append(value)
            mu_list.append(mu)
            sigma_list.append(sigma)

        x = self.norm(x)
        det_out = x[:, N_patch:]

        cls_logits = self.cls_head(det_out)
        bbox_pred = self.bbox_head(det_out).sigmoid()

        has_lp = log_prob_list[0] is not None
        return {
            'cls_logits': cls_logits,
            'bbox_pred': bbox_pred,
            'w_t': torch.stack(w_list, dim=1),
            'log_probs': torch.stack(log_prob_list, dim=1) if has_lp else None,
            'values': torch.stack(value_list, dim=1),
            'mu': torch.stack(mu_list, dim=1),
            'sigma': torch.stack(sigma_list, dim=1),
        }

## 3. COCO Dataset

In [ ]:
COCO_CATEGORY_IDS = [
    1,2,3,4,5,6,7,8,9,10,11,13,14,15,16,17,18,19,20,21,22,23,24,25,27,
    28,31,32,33,34,35,36,37,38,39,40,41,42,43,44,46,47,48,49,50,51,52,
    53,54,55,56,57,58,59,60,61,62,63,64,65,67,70,72,73,74,75,76,77,78,
    79,80,81,82,84,85,86,87,88,89,90
]
COCO_CAT_TO_IDX = {cat_id: idx for idx, cat_id in enumerate(COCO_CATEGORY_IDS)}
NUM_COCO_CLASSES = 80


class COCODetectionDataset(torch.utils.data.Dataset):
    """Wraps torchvision.CocoDetection. Resizes to (image_size, image_size),
    normalizes boxes to [0,1] as (cx, cy, w, h), maps COCO IDs to 0-79."""
    def __init__(self, img_dir, ann_file, image_size=144):
        self.coco = CocoDetection(img_dir, ann_file)
        self.image_size = image_size
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.coco)

    def __getitem__(self, idx):
        img, anns = self.coco[idx]
        orig_w, orig_h = img.size
        if img.mode != 'RGB':
            img = img.convert('RGB')
        img_tensor = self.transform(img)

        boxes, labels = [], []
        for ann in anns:
            if ann.get('iscrowd', 0):
                continue
            cat_id = ann['category_id']
            if cat_id not in COCO_CAT_TO_IDX:
                continue
            x, y, bw, bh = ann['bbox']
            if bw <= 0 or bh <= 0:
                continue
            cx = max(0.0, min(1.0, (x + bw / 2.0) / orig_w))
            cy = max(0.0, min(1.0, (y + bh / 2.0) / orig_h))
            nw = max(0.0, min(1.0, bw / orig_w))
            nh = max(0.0, min(1.0, bh / orig_h))
            boxes.append([cx, cy, nw, nh])
            labels.append(COCO_CAT_TO_IDX[cat_id])

        if len(boxes) == 0:
            boxes = torch.zeros(0, 4, dtype=torch.float32)
            labels = torch.zeros(0, dtype=torch.long)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.long)
        return img_tensor, {'boxes': boxes, 'labels': labels}


def coco_collate_fn(batch):
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets

## 4. Hungarian Matching & Detection Loss

In [ ]:
def box_cxcywh_to_xyxy(boxes):
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], dim=-1)


@torch.no_grad()
def hungarian_match(pred_logits, pred_boxes, targets, no_object_class):
    B, N_det, C = pred_logits.shape
    indices = []
    for b in range(B):
        gt_boxes = targets[b]['boxes']
        gt_labels = targets[b]['labels']
        M = len(gt_labels)
        if M == 0:
            indices.append((torch.tensor([], dtype=torch.long), torch.tensor([], dtype=torch.long)))
            continue
        gt_boxes = gt_boxes.to(pred_logits.device)
        gt_labels = gt_labels.to(pred_logits.device)
        pred_prob = pred_logits[b].softmax(-1)
        cls_cost = -pred_prob[:, gt_labels]
        l1_cost = torch.cdist(pred_boxes[b], gt_boxes, p=1)
        pred_xyxy = box_cxcywh_to_xyxy(pred_boxes[b])
        gt_xyxy = box_cxcywh_to_xyxy(gt_boxes)
        giou = generalized_box_iou(pred_xyxy, gt_xyxy)
        cost = 1.0 * cls_cost + 5.0 * l1_cost + 2.0 * (-giou)
        row_ind, col_ind = linear_sum_assignment(cost.detach().cpu().numpy())
        indices.append((torch.tensor(row_ind, dtype=torch.long), torch.tensor(col_ind, dtype=torch.long)))
    return indices


def detection_loss(pred_logits, pred_boxes, targets, num_classes,
                   lambda_cls=1.0, lambda_l1=5.0, lambda_giou=2.0, no_object_weight=0.1):
    """DETR-style set prediction loss. Returns (total_loss_scalar, per_sample_losses_tensor, info_dict)."""
    B, N_det, C = pred_logits.shape
    no_object_class = num_classes

    indices = hungarian_match(pred_logits, pred_boxes, targets, no_object_class)

    # --- Per-sample classification loss ---
    target_labels = torch.full((B, N_det), no_object_class, dtype=torch.long, device=pred_logits.device)
    for b, (pred_idx, gt_idx) in enumerate(indices):
        if len(pred_idx) > 0:
            gt_labels = targets[b]['labels'].to(pred_logits.device)
            target_labels[b, pred_idx] = gt_labels[gt_idx]

    weight = torch.ones(num_classes + 1, device=pred_logits.device)
    weight[no_object_class] = no_object_weight

    # Per-sample CE
    per_sample_cls = torch.zeros(B, device=pred_logits.device)
    for b in range(B):
        per_sample_cls[b] = F.cross_entropy(pred_logits[b], target_labels[b], weight=weight)

    # --- Per-sample box losses ---
    per_sample_box = torch.zeros(B, device=pred_logits.device)
    total_matched = 0
    for b, (pred_idx, gt_idx) in enumerate(indices):
        if len(pred_idx) == 0:
            continue
        gt_boxes = targets[b]['boxes'].to(pred_logits.device)
        matched_pred = pred_boxes[b][pred_idx]
        matched_gt = gt_boxes[gt_idx]
        l1 = F.l1_loss(matched_pred, matched_gt, reduction='mean')
        pred_xyxy = box_cxcywh_to_xyxy(matched_pred)
        gt_xyxy = box_cxcywh_to_xyxy(matched_gt)
        giou = generalized_box_iou(pred_xyxy, gt_xyxy)
        giou_loss = (1.0 - giou.diag()).mean()
        per_sample_box[b] = lambda_l1 * l1 + lambda_giou * giou_loss
        total_matched += len(pred_idx)

    per_sample_loss = lambda_cls * per_sample_cls + per_sample_box
    total_loss = per_sample_loss.mean()

    return total_loss, per_sample_loss, {
        'cls': per_sample_cls.mean().item(),
        'box': per_sample_box.mean().item(),
        'total': total_loss.item(),
        'matched': max(total_matched, 1)
    }

## 5. Anti-Reward-Hacking Regularizers & Squashed Entropy

Three regularizers fix the uniform w~0.5 collapse:

| Regularizer | Formula | Effect |
|---|---|---|
| **Bimodal Sparsity** | `w*(1-w)` minimize | Pushes each weight toward 0 or 1 |
| **Variance Maximization** | `-var(w, dim=tokens)` | Forces different weights across tokens |
| **Symmetric Budget** | `(mean(w) - target)^2` | Penalizes ANY deviation from target |

In [ ]:
def compute_squashed_entropy(sigma, w_raw_samples=None):
    """Jacobian-corrected entropy for the squashed Gaussian (Paper Eq. 13)."""
    gaussian_entropy = 0.5 * torch.log(2 * math.pi * math.e * sigma.pow(2) + 1e-8)
    if w_raw_samples is not None:
        tanh_z = torch.tanh(w_raw_samples)
        jacobian_correction = torch.log(0.5 * (1.0 - tanh_z.pow(2)) + 1e-5)
        return (gaussian_entropy + jacobian_correction).mean()
    else:
        return gaussian_entropy.mean()


def compute_routing_regularizers(w_t, target_budget=0.5,
                                  lambda_sparse=1.0, lambda_var=0.5, lambda_budget=1.0):
    """Compute anti-reward-hacking losses for routing weights w_t: (B, L, N)."""
    sparsity_loss = (w_t * (1.0 - w_t)).mean()
    per_layer_var = w_t.var(dim=-1)
    variance_loss = -per_layer_var.mean()
    budget_loss = (w_t.mean() - target_budget).pow(2)

    total_reg = (lambda_sparse * sparsity_loss +
                 lambda_var * variance_loss +
                 lambda_budget * budget_loss)
    return total_reg, {
        'sparsity': sparsity_loss.item(),
        'variance': per_layer_var.mean().item(),
        'budget_dev': (w_t.mean() - target_budget).abs().item(),
        'reg_total': total_reg.item(),
    }

## 6. Sanity Check & Evaluation

In [ ]:
def sanity_check_overfit(model, train_loader, device, epochs=50):
    """Overfit on 1 batch to verify the detection model can learn."""
    print("\n--- Pre-Flight Check: Single-Batch Overfit (Detection) ---")
    model.train()
    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.0)
    scaler = torch.amp.GradScaler(device.type, enabled=device.type == 'cuda')

    images, targets = next(iter(train_loader))
    images = images.to(device)

    loop = tqdm(range(epochs), desc="Overfitting Single Batch")
    for _ in loop:
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type):
            outputs = model(images, deterministic=True, phase1=True)
            loss, _, info = detection_loss(
                outputs['cls_logits'], outputs['bbox_pred'], targets, NUM_COCO_CLASSES)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loop.set_postfix(Loss=f"{loss.item():.4f}", Cls=f"{info['cls']:.4f}",
                         Box=f"{info['box']:.4f}", Matched=info['matched'])
    print(f"Final loss: {loss.item():.4f} -- if this decreased, the model can learn!")


@torch.no_grad()
def evaluate_detection(model, val_loader, device, epoch):
    model.eval()
    total_loss, total_steps = 0.0, 0
    print(f"\n--- Detection Validation (Epoch {epoch+1}) ---")
    for images, targets in tqdm(val_loader, desc="Evaluating", leave=False):
        images = images.to(device)
        with torch.autocast(device_type=device.type):
            outputs = model(images, deterministic=True)
            loss, _, _ = detection_loss(
                outputs['cls_logits'], outputs['bbox_pred'], targets, NUM_COCO_CLASSES)
        total_loss += loss.item()
        total_steps += 1
    avg = total_loss / max(total_steps, 1)
    w_t = outputs['w_t']
    print(f"Val Loss={avg:.4f} | w_mean={w_t.mean().item():.3f} | w_var={w_t.var(dim=-1).mean().item():.4f}\n")
    model.train()
    return avg

## 7. Stage 3: PPO Detection Fine-tuning with Anti-Reward-Hacking

The router is trained via **PPO** (reinforcement learning) with a reward that includes:
- **Task metric**: negative per-sample detection loss
- **Variance bonus**: reward high per-sample routing variance (anti-collapse)
- **Sparsity penalty**: penalize per-sample w*(1-w) (push to binary)
- **Symmetric budget**: penalize per-sample budget deviation squared

The detection loss also trains the backbone + detection heads supervised.  
The anti-reward-hacking terms appear in BOTH the PPO reward AND as differentiable loss terms.

In [ ]:
def train_stage_3_ppo_detection(model, train_loader, val_loader, epochs, device,
                                target_budget=0.5, lambda_penalty=2.0,
                                lambda_sparse=1.0, lambda_var=0.5,
                                ppo_epochs=3, clip_eps=0.2, entropy_coeff=0.01,
                                lr=1e-5):
    """
    Stage 3: End-to-end PPO fine-tuning for detection.

    Key fix from v2-classification: the PPO reward now includes
    anti-reward-hacking terms (variance bonus, sparsity penalty,
    symmetric budget) that prevent the uniform w~0.5 collapse.
    """
    print("\n" + "="*70)
    print("Stage 3: PPO Detection Fine-tuning with Anti-Reward-Hacking")
    print("="*70)
    model.train()

    optimizer = optim.AdamW(model.parameters(), lr=lr)
    scaler = torch.amp.GradScaler(device.type, enabled=device.type == 'cuda')

    for epoch in range(epochs):
        loop = tqdm(train_loader, desc=f"Stage 3 Epoch [{epoch+1}/{epochs}]")
        for images, targets in loop:
            images = images.to(device)

            # ===== STEP 1: Collect rollout (old policy) =====
            with torch.no_grad():
                with torch.autocast(device_type=device.type):
                    old_outputs = model(images, deterministic=False, use_dilution=True)
                    old_log_probs = old_outputs['log_probs'].detach()  # (B, L, N_patch)
                    old_values = old_outputs['values'].detach()        # (B, L, N_patch)
                    old_w_t = old_outputs['w_t'].detach()              # (B, L, N_patch)

                    # --- Per-sample detection reward ---
                    _, per_sample_det_loss, _ = detection_loss(
                        old_outputs['cls_logits'], old_outputs['bbox_pred'],
                        targets, NUM_COCO_CLASSES)
                    task_metric = -per_sample_det_loss              # (B,) higher = better

                    # --- Per-sample anti-reward-hacking terms ---
                    # Variance bonus: reward routing differentiation per sample
                    b_var = old_w_t.var(dim=-1).mean(dim=1)        # (B,)

                    # Sparsity penalty: penalize w*(1-w) per sample
                    b_sparse = (old_w_t * (1.0 - old_w_t)).mean(dim=(1, 2))  # (B,)

                    # Symmetric budget penalty per sample
                    b_used = old_w_t.mean(dim=(1, 2))              # (B,)
                    b_budget = (b_used - target_budget).pow(2)     # (B,)

                    # --- Terminal reward (per-sample) ---
                    terminal_reward = (task_metric
                                       + lambda_var * b_var
                                       - lambda_sparse * b_sparse
                                       - lambda_penalty * b_budget)   # (B,)

                    # Broadcast to all layers/tokens
                    returns = terminal_reward.view(-1, 1, 1).expand_as(old_values)

                    # GAE-style advantages
                    advantages = returns - old_values
                    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

            # ===== STEP 2: PPO policy update (K epochs) =====
            for k in range(ppo_epochs):
                optimizer.zero_grad()

                with torch.autocast(device_type=device.type):
                    new_outputs = model(images, deterministic=False, use_dilution=True)
                    new_log_probs = new_outputs['log_probs']
                    new_values = new_outputs['values']
                    new_sigma = new_outputs['sigma']
                    new_w_t = new_outputs['w_t']

                    # --- Detection loss (supervised, for backbone + det heads) ---
                    loss_det, _, det_info = detection_loss(
                        new_outputs['cls_logits'], new_outputs['bbox_pred'],
                        targets, NUM_COCO_CLASSES)

                    # --- PPO policy loss ---
                    ratio = torch.exp(new_log_probs - old_log_probs)
                    surr1 = ratio * advantages
                    surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * advantages
                    policy_loss = -torch.min(surr1, surr2).mean()

                    # --- Value loss ---
                    value_loss = F.mse_loss(new_values, returns)

                    # --- Squashed Gaussian entropy bonus ---
                    entropy = compute_squashed_entropy(new_sigma)
                    entropy_bonus = entropy_coeff * entropy

                    # --- Anti-reward-hacking as differentiable loss terms ---
                    loss_reg, reg_info = compute_routing_regularizers(
                        new_w_t, target_budget=target_budget,
                        lambda_sparse=lambda_sparse,
                        lambda_var=lambda_var,
                        lambda_budget=lambda_penalty)

                    # --- Total loss ---
                    loss_ppo = (policy_loss
                                + 0.5 * value_loss
                                + loss_reg
                                - entropy_bonus)
                    total_loss = loss_det + loss_ppo

                scaler.scale(total_loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

            loop.set_postfix(
                det=f"{det_info['total']:.3f}",
                PPO=f"{policy_loss.item():.3f}",
                spar=f"{reg_info['sparsity']:.3f}",
                wvar=f"{reg_info['variance']:.4f}",
                wmean=f"{new_w_t.mean().item():.3f}",
            )

        if (epoch + 1) % 2 == 0 or (epoch + 1) == epochs:
            evaluate_detection(model, val_loader, device, epoch)

## 8. Visualization

In [ ]:
@torch.no_grad()
def visualize_detection_routing(model, dataset, device, image_index=0):
    model.eval()
    img_tensor, target = dataset[image_index]
    image_batch = img_tensor.unsqueeze(0).to(device)

    outputs = model(image_batch, deterministic=True)
    w_t = outputs['w_t']
    cls_logits = outputs['cls_logits']
    bbox_pred = outputs['bbox_pred']

    routing_weights = w_t[0].mean(dim=0).cpu().numpy()
    grid_size = int(math.sqrt(len(routing_weights)))
    heatmap = routing_weights.reshape(grid_size, grid_size)

    img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_np = np.clip(std * img_np + mean, 0, 1)

    img_size = dataset.image_size
    heatmap_resized = cv2.resize(heatmap, (img_size, img_size), interpolation=cv2.INTER_CUBIC)

    probs = cls_logits[0].softmax(-1)
    scores, pred_labels = probs[:, :-1].max(-1)
    keep = scores > 0.3
    kept_boxes = bbox_pred[0][keep].cpu().numpy()
    kept_scores = scores[keep].cpu().numpy()
    kept_labels = pred_labels[keep].cpu().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(img_np)
    for box in target['boxes'].numpy():
        cx, cy, w, h = box
        rect = plt.Rectangle(((cx-w/2)*img_size, (cy-h/2)*img_size), w*img_size, h*img_size,
                              linewidth=2, edgecolor='lime', facecolor='none')
        axes[0].add_patch(rect)
    axes[0].set_title(f"Ground Truth ({len(target['boxes'])} objects)", fontweight='bold')
    axes[0].axis('off')

    im = axes[1].imshow(heatmap_resized, cmap='jet', alpha=0.8, vmin=0, vmax=1)
    axes[1].set_title(f"Routing Map\nvar={w_t.var(dim=-1).mean().item():.4f}")
    axes[1].axis('off')
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    axes[2].imshow(img_np)
    axes[2].imshow(heatmap_resized, cmap='jet', alpha=0.4, vmin=0, vmax=1)
    for i, box in enumerate(kept_boxes):
        cx, cy, w, h = box
        rect = plt.Rectangle(((cx-w/2)*img_size, (cy-h/2)*img_size), w*img_size, h*img_size,
                              linewidth=2, edgecolor='yellow', facecolor='none')
        axes[2].add_patch(rect)
        cat_name = COCO_CATEGORY_IDS[kept_labels[i]] if kept_labels[i] < 80 else '?'
        axes[2].text((cx-w/2)*img_size, (cy-h/2)*img_size - 2, f"{cat_name}:{kept_scores[i]:.2f}",
                     color='yellow', fontsize=7, backgroundcolor='black')
    axes[2].set_title(f"Predictions ({len(kept_boxes)} detected)")
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()

## 9. Download Dataset & Setup

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
os.environ["KAGGLEHUB_CACHE"] = "D:/kagglehub_cache"

print("Downloading MSCOCO 2017 via kagglehub...")
coco_path = kagglehub.dataset_download("awsaf49/coco-2017-dataset")
print(f"Dataset path: {coco_path}")
coco_root = os.path.join(coco_path, "coco2017")
if not os.path.isdir(coco_root):
    coco_root = coco_path

train_img_dir = os.path.join(coco_root, "train2017")
val_img_dir = os.path.join(coco_root, "val2017")
train_ann = os.path.join(coco_root, "annotations", "instances_train2017.json")
val_ann = os.path.join(coco_root, "annotations", "instances_val2017.json")

for p, name in [(train_img_dir, "train images"), (val_img_dir, "val images"),
                (train_ann, "train annotations"), (val_ann, "val annotations")]:
    status = "[OK]" if os.path.exists(p) else "[FAIL]"
    print(f"  {status} {name}: {p}")

In [ ]:
print("Creating COCO detection datasets...")
train_dataset = COCODetectionDataset(train_img_dir, train_ann, image_size=144)
val_dataset = COCODetectionDataset(val_img_dir, val_ann, image_size=144)
print(f"Train: {len(train_dataset)} images | Val: {len(val_dataset)} images")

batch_size = 16
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=0, pin_memory=True, collate_fn=coco_collate_fn)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=0, pin_memory=True, collate_fn=coco_collate_fn)

## 10. Create Model & Load Stage 2 Weights

In [ ]:
model = ViTDetection(
    image_size=144, patch_size=12,
    d_model=256, depth=16, head=8, chunk_size=16,
    num_det_tokens=32,
    num_det_classes=NUM_COCO_CLASSES + 1  # 80 + 1 no-object = 81
).to(device)

total_params = sum(p.numel() for p in model.parameters())
det_params = sum(p.numel() for n, p in model.named_parameters()
                 if any(k in n for k in ['det_tokens', 'cls_head', 'bbox_head']))
print(f"Total params:     {total_params:,}")
print(f"Detection params: {det_params:,}")
print(f"Backbone+Router:  {total_params - det_params:,}")

stage2_path = "vit_v2_stage_2_router.pth"
if os.path.exists(stage2_path):
    model.load_backbone_weights(stage2_path, device=device)
else:
    print(f"Stage 2 checkpoint not found at {stage2_path}")
    print("   Training from scratch (no pre-trained backbone).")

## 11. Sanity Check

In [ ]:
sanity_check_overfit(model, train_loader, device, epochs=200)

## 12. Train Stage 3: PPO Detection + Anti-Reward-Hacking

In [ ]:
train_stage_3_ppo_detection(
    model, train_loader, val_loader,
    epochs=10,
    device=device,
    target_budget=0.5,
    lambda_penalty=2.0,    # symmetric budget penalty weight
    lambda_sparse=1.0,     # push w toward 0 or 1
    lambda_var=0.5,        # encourage spatial variance
    ppo_epochs=1,
    clip_eps=0.2,
    entropy_coeff=0.01,
    lr=1e-5,
)

torch.save(model.state_dict(), "vit_v2_stage_3_detection.pth")
print("Stage 3 detection checkpoint saved.")

## 13. Visualize Routing + Detections

In [ ]:
for idx in [0, 50, 100]:
    try:
        visualize_detection_routing(model, val_dataset, device, image_index=idx)
    except Exception as e:
        print(f"  Visualization failed for idx={idx}: {e}")